# Forest Cover Type — Revised Experimental Protocol


# Revision Protocol for Wasserstein Fidelity

This notebook implements the revised protocol used to address the reviewer comments.

**Core rules**

1. The train/validation/test split is created **before** scaling.
2. `MinMaxScaler` is fit on the **training split only**.
3. GANs are trained on the training split only.
4. Generator fidelity is evaluated using **synthetic-only observations versus held-out real validation observations** from the same class.
5. The reported fidelity diagnostic is the **mean marginal 1-Wasserstein discrepancy**
   \[
   \widehat d_y^{\mathrm{marg}}
   =\frac1d\sum_{j=1}^d W_1(\widehat P_{y,j}^{val},\widehat P_{G,y,j}).
   \]
   It is an empirical marginal diagnostic, **not** the joint multivariate Wasserstein distance in the theorem.
6. All fidelity calculations are made in the common `[0,1]` training-scaled feature space.
7. `LAMBDA_GRID = {0.25,0.50,0.75,1.00}` denotes the experimental **balancing strength**. It is kept distinct from the theoretical synthetic mixture proportion
   \[
   \alpha_y=\frac{n_{G,y}}{n_y+n_{G,y}}.
   \]
8. For each generative method/run, a fixed-size synthetic fidelity sample is generated independently of balancing strength. Thus generator fidelity \(\widehat d_y^{marg}\) is separated from the augmentation-weighted term.
9. The empirical analogue of the Wasserstein component in Theorem 1 (without the unknown Lipschitz constant) is saved as
   \[
   \widehat\Delta_A/L_\ell
   =\sum_y \widetilde\pi_y\alpha_y\widehat d_y^{marg}.
   \]
10. Test data are used only for final classifier evaluation. Validation data are used for fidelity estimation and can later be used for augmentation-strength selection.
11. Raw run-level outputs are saved so that replication-based uncertainty/statistical comparisons can be performed later.

`N_RUNS` is set to 10 by default. Increase it if desired after checking compute time.
\n**Additional safeguards:** Validation Macro-F1 selects augmentation strength separately for each run/method/classifier (ties choose smaller strength). Test results for the selected strengths are reported separately from the full fixed-strength sensitivity grid. Synthetic wilderness/soil categorical groups are repaired to valid one-hot values.\n
**Fix (October 2026):** Corrected ROS/SMOTE validation logging. Added checks for 34 validation rows and 8 selected augmentation strengths per run.


In [1]:
# ============================================================
# FOREST COVER — REVISION-READY EXPERIMENT
# ============================================================
import os, random, warnings, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive

drive.mount("/content/drive")

# ---------------- SETTINGS ----------------
DATA_PATH = "/content/drive/MyDrive/covtype.data.gz"
OUT_DIR = "/content/drive/MyDrive/CGA Ex/revision_protocol"
os.makedirs(OUT_DIR, exist_ok=True)

BASE_SEED = 44
N_RUNS = 10
LAMBDA_GRID = [0.25, 0.50, 0.75, 1.00]

SUBSAMPLE_N = 50000
TEST_SIZE = 0.20
VAL_SIZE = 0.20
LATENT_DIM = 64
BATCH_SIZE = 256
EPOCHS_CGAN = 300
EPOCHS_WGAN = 300
N_CRITIC = 5
GP_WEIGHT = 10.0
FIDELITY_N_PER_CLASS = 1000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ---------------- DATA ----------------
df = pd.read_csv(DATA_PATH, header=None, compression="gzip")
X_full = df.iloc[:, :-1].to_numpy(dtype=np.float32)
y_raw = df.iloc[:, -1].to_numpy()

le = LabelEncoder()
y_full = le.fit_transform(y_raw)
class_names = [
    "Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine",
    "Cottonwood/Willow", "Aspen", "Douglas Fir", "Krummholz"
]
n_classes = len(class_names)

Mounted at /content/drive
Device: cuda


In [ ]:



# Fixed stratified study subset, then run-specific train/val/test splits.
if SUBSAMPLE_N is not None and SUBSAMPLE_N < len(y_full):
    X_raw, _, y, _ = train_test_split(
        X_full, y_full, train_size=SUBSAMPLE_N,
        stratify=y_full, random_state=BASE_SEED
    )
else:
    X_raw, y = X_full, y_full

# Covtype: first 10 variables are quantitative; remaining 44 are binary indicators.
BINARY_IDX = np.arange(10, X_raw.shape[1])

def split_scale(X, y, seed):
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed
    )
    val_frac_within_train = VAL_SIZE / (1.0 - TEST_SIZE)
    X_train_raw, X_val_raw, y_train, y_val = train_test_split(
        X_train_raw, y_train, test_size=val_frac_within_train,
        stratify=y_train, random_state=seed + 10000
    )
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val = scaler.transform(X_val_raw).astype(np.float32)
    X_test = scaler.transform(X_test_raw).astype(np.float32)
    return X_train, X_val, X_test, y_train, y_val, y_test, scaler

def postprocess_covtype(X_syn):
    X_syn = np.clip(np.asarray(X_syn, dtype=np.float32), 0.0, 1.0)
    # Wilderness areas (10:14) and soil types (14:54) are one-hot groups.
    # Independent thresholding can create impossible rows with multiple/no categories.
    for left, right in [(10, 14), (14, 54)]:
        block = X_syn[:, left:right]
        winner = np.argmax(block, axis=1)
        X_syn[:, left:right] = 0.0
        X_syn[np.arange(len(X_syn)), left + winner] = 1.0
    return X_syn

def mean_marginal_w1(X_real, X_syn):
    if len(X_real) == 0 or len(X_syn) == 0:
        return np.nan
    return float(np.mean([
        wasserstein_distance(X_real[:, j], X_syn[:, j])
        for j in range(X_real.shape[1])
    ]))

def target_counts(y_train, balancing_strength):
    counts = pd.Series(y_train).value_counts().sort_index()
    n_max = counts.max()
    return {
        int(cls): int(round(n + balancing_strength * (n_max - n)))
        for cls, n in counts.items()
    }

def get_classifiers(seed):
    return {
        "RF": RandomForestClassifier(n_estimators=150, random_state=seed, n_jobs=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(64,64), max_iter=300, random_state=seed)
    }

def evaluate(Xtr, ytr, Xtest, ytest, seed, Xval=None, yval=None):
    """Fit each classifier once; evaluate on test and optionally validation."""
    rows, class_rows, val_rows = [], [], []
    for name, clf in get_classifiers(seed).items():
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xtest)
        rows.append({
            "Classifier": name,
            "Macro_F1": f1_score(ytest, pred, average="macro"),
            "Macro_Recall": recall_score(ytest, pred, average="macro"),
            "Balanced_Accuracy": balanced_accuracy_score(ytest, pred),
            "Accuracy": accuracy_score(ytest, pred)
        })
        rec = recall_score(ytest, pred, average=None, labels=np.arange(n_classes), zero_division=0)
        for cls, val in enumerate(rec):
            class_rows.append({
                "Classifier": name, "Class_ID": cls,
                "Class": class_names[cls], "Recall": val
            })
        if Xval is not None:
            val_pred = clf.predict(Xval)
            val_rows.append({
                "Classifier": name,
                "Val_Macro_F1": f1_score(yval, val_pred, average="macro"),
                "Val_Macro_Recall": recall_score(yval, val_pred, average="macro"),
                "Val_Balanced_Accuracy": balanced_accuracy_score(yval, val_pred),
                "Val_Accuracy": accuracy_score(yval, val_pred)
            })
    return pd.DataFrame(rows), pd.DataFrame(class_rows), pd.DataFrame(val_rows)

def random_oversample_to_targets(X, y, targets, seed):
    rng = np.random.default_rng(seed)
    xs, ys = [X], [y]
    for cls, target in targets.items():
        idx = np.where(y == cls)[0]
        n_add = max(0, target - len(idx))
        if n_add:
            add = rng.choice(idx, size=n_add, replace=True)
            xs.append(X[add]); ys.append(np.full(n_add, cls))
    return np.vstack(xs), np.concatenate(ys)

def smote_to_targets(X, y, targets, seed):
    strategy = {cls: target for cls, target in targets.items()
                if target > np.sum(y == cls)}
    if not strategy:
        return X.copy(), y.copy()
    return SMOTE(sampling_strategy=strategy, random_state=seed, k_neighbors=5).fit_resample(X, y)

# ---------------- MODELS ----------------
class Generator(nn.Module):
    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(n_classes, n_classes)
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM + n_classes, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, feature_dim), nn.Sigmoid()
        )
    def forward(self, z, y):
        return self.net(torch.cat([z, self.emb(y)], dim=1))

class Discriminator(nn.Module):
    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(n_classes, n_classes)
        self.net = nn.Sequential(
            nn.Linear(feature_dim + n_classes, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
        return self.net(torch.cat([x, self.emb(y)], dim=1))

class Critic(nn.Module):
    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(n_classes, n_classes)
        self.net = nn.Sequential(
            nn.Linear(feature_dim + n_classes, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x, y):
        return self.net(torch.cat([x, self.emb(y)], dim=1))

def balanced_batch(Xt, yt, batch_size):
    per = max(1, batch_size // n_classes)
    xs, ys = [], []
    for cls in range(n_classes):
        idx = torch.where(yt == cls)[0]
        choose = idx[torch.randint(0, len(idx), (per,), device=DEVICE)]
        xs.append(Xt[choose]); ys.append(yt[choose])
    x = torch.cat(xs); lab = torch.cat(ys)
    if len(lab) < batch_size:
        extra = torch.randint(0, len(yt), (batch_size-len(lab),), device=DEVICE)
        x = torch.cat([x, Xt[extra]]); lab = torch.cat([lab, yt[extra]])
    perm = torch.randperm(len(lab), device=DEVICE)
    return x[perm], lab[perm]

def train_cgan(X_train, y_train, seed):
    set_seed(seed)
    Xt = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    G = Generator(X_train.shape[1], n_classes).to(DEVICE)
    D = Discriminator(X_train.shape[1], n_classes).to(DEVICE)
    optG = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
    optD = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
    bce = nn.BCELoss()

    for _ in range(EPOCHS_CGAN):
        real, labels = balanced_batch(Xt, yt, BATCH_SIZE)
        z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
        fake = G(z, labels)

        optD.zero_grad()
        dloss = bce(D(real, labels), torch.ones((len(labels),1), device=DEVICE)) + \
                bce(D(fake.detach(), labels), torch.zeros((len(labels),1), device=DEVICE))
        dloss.backward(); optD.step()

        z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
        optG.zero_grad()
        fake = G(z, labels)
        gloss = bce(D(fake, labels), torch.ones((len(labels),1), device=DEVICE))
        gloss.backward(); optG.step()
    return G

def gradient_penalty(C, real, fake, labels):
    eps = torch.rand(len(real), 1, device=DEVICE).expand_as(real)
    interp = eps*real + (1-eps)*fake
    interp.requires_grad_(True)
    score = C(interp, labels)
    grad = torch.autograd.grad(
        outputs=score, inputs=interp, grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    return ((grad.view(len(real), -1).norm(2, dim=1)-1)**2).mean()

def train_cwgan_gp(X_train, y_train, seed):
    set_seed(seed)
    Xt = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    G = Generator(X_train.shape[1], n_classes).to(DEVICE)
    C = Critic(X_train.shape[1], n_classes).to(DEVICE)
    optG = optim.Adam(G.parameters(), lr=1e-4, betas=(0.5,0.9))
    optC = optim.Adam(C.parameters(), lr=1e-4, betas=(0.5,0.9))

    for _ in range(EPOCHS_WGAN):
        for _ in range(N_CRITIC):
            real, labels = balanced_batch(Xt, yt, BATCH_SIZE)
            z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
            fake = G(z, labels)
            optC.zero_grad()
            lossC = C(fake.detach(), labels).mean() - C(real, labels).mean()
            lossC = lossC + GP_WEIGHT*gradient_penalty(C, real, fake.detach(), labels)
            lossC.backward(); optC.step()

        _, labels = balanced_batch(Xt, yt, BATCH_SIZE)
        z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
        optG.zero_grad()
        fake = G(z, labels)
        lossG = -C(fake, labels).mean()
        lossG.backward(); optG.step()
    return G

def generate_by_counts(G, counts_to_generate):
    G.eval()
    xs, ys = [], []
    with torch.no_grad():
        for cls, n in counts_to_generate.items():
            if n <= 0:
                continue
            labels = torch.full((n,), int(cls), dtype=torch.long, device=DEVICE)
            z = torch.randn(n, LATENT_DIM, device=DEVICE)
            x = G(z, labels).cpu().numpy()
            xs.append(postprocess_covtype(x))
            ys.append(np.full(n, cls))
    if not xs:
        return np.empty((0, X_raw.shape[1]), np.float32), np.empty((0,), int)
    return np.vstack(xs), np.concatenate(ys)

# ---------------- MAIN EXPERIMENT ----------------
metric_rows, class_recall_rows, fidelity_rows, validation_rows = [], [], [], []

for run in range(N_RUNS):
    seed = BASE_SEED + run
    set_seed(seed)

    X_train, X_val, X_test, y_train, y_val, y_test, scaler = split_scale(X_raw, y, seed)
    train_counts = pd.Series(y_train).value_counts().sort_index().to_dict()

    base, base_cls, base_val = evaluate(X_train, y_train, X_test, y_test, seed, X_val, y_val)
    base["RunID"], base["Method"], base["Balancing_Strength"] = run, "Real", 0.0
    base_cls["RunID"], base_cls["Method"], base_cls["Balancing_Strength"] = run, "Real", 0.0
    metric_rows.append(base); class_recall_rows.append(base_cls)
    base_val["RunID"], base_val["Method"], base_val["Balancing_Strength"] = run, "Real", 0.0
    validation_rows.append(base_val)

    print(f"Run {run+1}/{N_RUNS}: training CGAN")
    cgan = train_cgan(X_train, y_train, seed)
    print(f"Run {run+1}/{N_RUNS}: training CWGAN-GP")
    cwgan = train_cwgan_gp(X_train, y_train, seed)

    # Fixed fidelity samples by class, independent of balancing strength.
    fixed_d = {}
    for method, G in [("CGAN", cgan), ("CWGAN-GP", cwgan)]:
        for cls in range(n_classes):
            n_fid = max(FIDELITY_N_PER_CLASS, int(np.sum(y_val == cls)))
            X_syn, y_syn = generate_by_counts(G, {cls: n_fid})
            fixed_d[(method, cls)] = mean_marginal_w1(
                X_val[y_val == cls], X_syn[y_syn == cls]
            )

    for lam in LAMBDA_GRID:
        targets = target_counts(y_train, lam)

        X_ros, y_ros = random_oversample_to_targets(X_train, y_train, targets, seed)
        tmp, tmp_cls, tmp_val = evaluate(X_ros, y_ros, X_test, y_test, seed, X_val, y_val)
        tmp["RunID"], tmp["Method"], tmp["Balancing_Strength"] = run, "ROS", lam
        tmp_cls["RunID"], tmp_cls["Method"], tmp_cls["Balancing_Strength"] = run, "ROS", lam
        metric_rows.append(tmp); class_recall_rows.append(tmp_cls)
        tmp_val["RunID"], tmp_val["Method"], tmp_val["Balancing_Strength"] = run, "ROS", lam
        validation_rows.append(tmp_val.copy())

        X_sm, y_sm = smote_to_targets(X_train, y_train, targets, seed)
        tmp, tmp_cls, tmp_val = evaluate(X_sm, y_sm, X_test, y_test, seed, X_val, y_val)
        tmp["RunID"], tmp["Method"], tmp["Balancing_Strength"] = run, "SMOTE", lam
        tmp_cls["RunID"], tmp_cls["Method"], tmp_cls["Balancing_Strength"] = run, "SMOTE", lam
        metric_rows.append(tmp); class_recall_rows.append(tmp_cls)
        tmp_val["RunID"], tmp_val["Method"], tmp_val["Balancing_Strength"] = run, "SMOTE", lam
        validation_rows.append(tmp_val.copy())

        gen_counts = {cls: max(0, targets[cls] - train_counts[cls]) for cls in targets}

        for method, G in [("CGAN", cgan), ("CWGAN-GP", cwgan)]:
            X_syn, y_syn = generate_by_counts(G, gen_counts)
            X_aug = np.vstack([X_train, X_syn])
            y_aug = np.concatenate([y_train, y_syn])

            tmp, tmp_cls, tmp_val = evaluate(X_aug, y_aug, X_test, y_test, seed, X_val, y_val)
            tmp["RunID"], tmp["Method"], tmp["Balancing_Strength"] = run, method, lam
            tmp_cls["RunID"], tmp_cls["Method"], tmp_cls["Balancing_Strength"] = run, method, lam
            metric_rows.append(tmp); class_recall_rows.append(tmp_cls)
            tmp_val["RunID"], tmp_val["Method"], tmp_val["Balancing_Strength"] = run, method, lam
            validation_rows.append(tmp_val)

            N_aug = len(y_train) + len(y_syn)
            for cls in range(n_classes):
                n_real = int(train_counts[cls])
                n_gen = int(gen_counts[cls])
                alpha_y = 0.0 if (n_real+n_gen)==0 else n_gen/(n_real+n_gen)
                pi_tilde_y = (n_real+n_gen)/N_aug
                d_hat = fixed_d[(method, cls)]
                fidelity_rows.append({
                    "RunID": run, "Seed": seed, "Method": method,
                    "Balancing_Strength": lam, "Class_ID": cls,
                    "Class": class_names[cls],
                    "n_real_train": n_real, "n_generated": n_gen,
                    "alpha_y": alpha_y, "pi_tilde_y": pi_tilde_y,
                    "Mean_Marginal_W1": d_hat,
                    "Weighted_W1_Component": pi_tilde_y*alpha_y*d_hat
                })

    # run checkpoint
    pd.concat(metric_rows, ignore_index=True).to_csv(f"{OUT_DIR}/forest_metrics_checkpoint.csv", index=False)
    pd.concat(class_recall_rows, ignore_index=True).to_csv(f"{OUT_DIR}/forest_class_recall_checkpoint.csv", index=False)
    pd.DataFrame(fidelity_rows).to_csv(f"{OUT_DIR}/forest_fidelity_checkpoint.csv", index=False)
    pd.concat(validation_rows, ignore_index=True).to_csv(f"{OUT_DIR}/forest_validation_checkpoint.csv", index=False)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Check the full per-seed validation grid before selecting lambda.
validation = pd.concat(validation_rows, ignore_index=True)
expected_methods = {"Real", "ROS", "SMOTE", "CGAN", "CWGAN-GP"}
assert len(validation) == N_RUNS * (2 + 4 * len(LAMBDA_GRID) * 2), (
    f"Expected {N_RUNS * 34} validation rows; found {len(validation)}"
)
assert set(validation["Method"]) == expected_methods
assert not validation.duplicated(["RunID", "Method", "Classifier", "Balancing_Strength"]).any(), (
    "Duplicate method/classifier/strength combinations in validation results"
)
assert (validation["Method"].value_counts().to_dict() == {
    "Real": N_RUNS * 2,
    "ROS": N_RUNS * 2 * len(LAMBDA_GRID),
    "SMOTE": N_RUNS * 2 * len(LAMBDA_GRID),
    "CGAN": N_RUNS * 2 * len(LAMBDA_GRID),
    "CWGAN-GP": N_RUNS * 2 * len(LAMBDA_GRID),
}), "Missing or duplicated method rows in validation results"

# Select strength using validation Macro-F1 only; tie-break toward smaller lambda.
validation.to_csv(f"{OUT_DIR}/forest_validation_raw.csv", index=False)
candidates = validation[validation["Method"] != "Real"].copy()
selected = (candidates.sort_values(
    ["RunID", "Method", "Classifier", "Val_Macro_F1", "Balancing_Strength"],
    ascending=[True, True, True, False, True]
).drop_duplicates(["RunID", "Method", "Classifier"]))
selected = selected.rename(columns={"Balancing_Strength": "Selected_Lambda"})
selected = selected[["RunID", "Method", "Classifier", "Selected_Lambda",
                     "Val_Macro_F1", "Val_Macro_Recall", "Val_Accuracy"]]
assert len(selected) == N_RUNS * 4 * 2, "Expected 8 selected lambdas per run"
selected.to_csv(f"{OUT_DIR}/forest_selected_lambda_raw.csv", index=False)

metrics = pd.concat(metric_rows, ignore_index=True)
class_recall = pd.concat(class_recall_rows, ignore_index=True)
fidelity = pd.DataFrame(fidelity_rows)

# Aggregate theorem-aligned Wasserstein component:
delta_component = (
    fidelity.groupby(["RunID","Seed","Method","Balancing_Strength"], as_index=False)
    ["Weighted_W1_Component"].sum()
    .rename(columns={"Weighted_W1_Component":"Delta_Component_over_L"})
)

metrics.to_csv(f"{OUT_DIR}/forest_metrics_raw.csv", index=False)
selected_test = selected.merge(
    metrics, left_on=["RunID", "Method", "Classifier", "Selected_Lambda"],
    right_on=["RunID", "Method", "Classifier", "Balancing_Strength"],
    how="left", validate="one_to_one"
)
assert len(selected_test) == N_RUNS * 4 * 2 and selected_test["Macro_F1"].notna().all()
selected_test.to_csv(f"{OUT_DIR}/forest_validation_selected_test_raw.csv", index=False)
selected_summary = selected_test.groupby(["Method", "Classifier"], as_index=False).agg(
    Macro_F1_mean=("Macro_F1", "mean"),
    Macro_F1_sd=("Macro_F1", "std"),
    Macro_Recall_mean=("Macro_Recall", "mean"),
    Macro_Recall_sd=("Macro_Recall", "std"),
    Accuracy_mean=("Accuracy", "mean"),
    Accuracy_sd=("Accuracy", "std")
)
selected_summary.to_csv(f"{OUT_DIR}/forest_validation_selected_test_summary.csv", index=False)
class_recall.to_csv(f"{OUT_DIR}/forest_class_recall_raw.csv", index=False)
fidelity.to_csv(f"{OUT_DIR}/forest_fidelity_raw.csv", index=False)
delta_component.to_csv(f"{OUT_DIR}/forest_delta_component_raw.csv", index=False)

display(metrics.head())
display(fidelity.head())
display(delta_component.head())


Mounted at /content/drive
Device: cuda
Run 1/10: training CGAN
Run 1/10: training CWGAN-GP
Run 2/10: training CGAN
Run 2/10: training CWGAN-GP
Run 3/10: training CGAN
Run 3/10: training CWGAN-GP
Run 4/10: training CGAN
Run 4/10: training CWGAN-GP
Run 5/10: training CGAN
Run 5/10: training CWGAN-GP
Run 6/10: training CGAN
Run 6/10: training CWGAN-GP
Run 7/10: training CGAN
Run 7/10: training CWGAN-GP
Run 8/10: training CGAN
Run 8/10: training CWGAN-GP
Run 9/10: training CGAN
Run 9/10: training CWGAN-GP
Run 10/10: training CGAN
Run 10/10: training CWGAN-GP


,Classifier,Macro_F1,Macro_Recall,Balanced_Accuracy,Accuracy,RunID,Method,Balancing_Strength
0,RF,0.768403,0.720141,0.720141,0.8656,0,Real,0.00
1,MLP,0.708693,0.694503,0.694503,0.8116,0,Real,0.00
2,RF,0.800587,0.768627,0.768627,0.8743,0,ROS,0.25
3,MLP,0.736924,0.775939,0.775939,0.8012,0,ROS,0.25
4,RF,0.803199,0.801875,0.801875,0.8721,0,SMOTE,0.25


,RunID,Seed,Method,Balancing_Strength,Class_ID,Class,n_real_train,n_generated,alpha_y,pi_tilde_y,Mean_Marginal_W1,Weighted_W1_Component
0,0,44,CGAN,0.25,0,Spruce/Fir,10938,922,0.077740,0.246575,0.103897,0.001992
1,0,44,CGAN,0.25,1,Lodgepole Pine,14628,0,0.000000,0.304123,0.096584,0.000000
2,0,44,CGAN,0.25,2,Ponderosa Pine,1846,3196,0.633875,0.104825,0.112003,0.007442
3,0,44,CGAN,0.25,3,Cottonwood/Willow,142,3622,0.962274,0.078255,0.121848,0.009176
4,0,44,CGAN,0.25,4,Aspen,490,3534,0.878231,0.083661,0.104725,0.007694


,RunID,Seed,Method,Balancing_Strength,Delta_Component_over_L
0,0,44,CGAN,0.25,0.041563
1,0,44,CGAN,0.50,0.060398
2,0,44,CGAN,0.75,0.071145
3,0,44,CGAN,1.00,0.078093
4,0,44,CWGAN-GP,0.25,0.028569


## Reporting summaries
These summaries do not replace the run-level raw files used for inference.

In [ ]:

metric_summary = (
    metrics.groupby(["Method","Balancing_Strength","Classifier"], as_index=False)
    .agg(
        Macro_F1_mean=("Macro_F1","mean"),
        Macro_F1_sd=("Macro_F1","std"),
        Macro_Recall_mean=("Macro_Recall","mean"),
        Macro_Recall_sd=("Macro_Recall","std"),
        Accuracy_mean=("Accuracy","mean"),
        Accuracy_sd=("Accuracy","std")
    )
)

fidelity_summary = (
    fidelity.groupby(["Method","Balancing_Strength","Class_ID","Class"], as_index=False)
    .agg(
        Mean_Marginal_W1_mean=("Mean_Marginal_W1","mean"),
        Mean_Marginal_W1_sd=("Mean_Marginal_W1","std"),
        Weighted_W1_Component_mean=("Weighted_W1_Component","mean"),
        Weighted_W1_Component_sd=("Weighted_W1_Component","std"),
        alpha_y_mean=("alpha_y","mean"),
        pi_tilde_y_mean=("pi_tilde_y","mean")
    )
)

delta_summary = (
    delta_component.groupby(["Method","Balancing_Strength"], as_index=False)
    .agg(
        Delta_Component_mean=("Delta_Component_over_L","mean"),
        Delta_Component_sd=("Delta_Component_over_L","std")
    )
)

metric_summary.to_csv(f"{OUT_DIR}/forest_metrics_summary.csv", index=False)
fidelity_summary.to_csv(f"{OUT_DIR}/forest_fidelity_summary.csv", index=False)
delta_summary.to_csv(f"{OUT_DIR}/forest_delta_component_summary.csv", index=False)

display(metric_summary)
display(fidelity_summary)
display(delta_summary)


,Method,Balancing_Strength,Classifier,Macro_F1_mean,Macro_F1_sd,Macro_Recall_mean,Macro_Recall_sd,Accuracy_mean,Accuracy_sd
0,CGAN,0.25,MLP,0.695944,0.017289,0.667152,0.023405,0.80776,0.006094
1,CGAN,0.25,RF,0.754514,0.013265,0.706532,0.014737,0.86141,0.005193
2,CGAN,0.50,MLP,0.697390,0.015017,0.672571,0.022213,0.80628,0.007713
3,CGAN,0.50,RF,0.756137,0.015158,0.707524,0.015616,0.86131,0.005117
4,CGAN,0.75,MLP,0.697904,0.017425,0.674540,0.026885,0.80685,0.013131
5,CGAN,0.75,RF,0.754182,0.014036,0.706095,0.015450,0.86123,0.005249
6,CGAN,1.00,MLP,0.697029,0.023530,0.674298,0.033740,0.80337,0.012361
7,CGAN,1.00,RF,0.755927,0.015279,0.707544,0.016934,0.86155,0.005059
8,CWGAN-GP,0.25,MLP,0.691172,0.013859,0.665393,0.017235,0.80674,0.009820
9,CWGAN-GP,0.25,RF,0.751146,0.012316,0.700884,0.013920,0.86103,0.003366


,Method,Balancing_Strength,Class_ID,Class,Mean_Marginal_W1_mean,Mean_Marginal_W1_sd,Weighted_W1_Component_mean,Weighted_W1_Component_sd,alpha_y_mean,pi_tilde_y_mean
0,CGAN,0.25,0,Spruce/Fir,0.102017,0.010884,0.001956,0.000209,0.077740,0.246575
1,CGAN,0.25,1,Lodgepole Pine,0.101252,0.012145,0.000000,0.000000,0.000000,0.304123
2,CGAN,0.25,2,Ponderosa Pine,0.101874,0.010344,0.006769,0.000687,0.633875,0.104825
3,CGAN,0.25,3,Cottonwood/Willow,0.107295,0.019270,0.008080,0.001451,0.962274,0.078255
4,CGAN,0.25,4,Aspen,0.100620,0.010625,0.007393,0.000781,0.878231,0.083661
5,CGAN,0.25,5,Douglas Fir,0.100433,0.009792,0.007168,0.000699,0.792841,0.090023
6,CGAN,0.25,6,Krummholz,0.103854,0.010617,0.007324,0.000749,0.762076,0.092538
7,CGAN,0.50,0,Spruce/Fir,0.102017,0.010884,0.002843,0.000303,0.144332,0.193103
8,CGAN,0.50,1,Lodgepole Pine,0.101252,0.012145,0.000000,0.000000,0.000000,0.220973
9,CGAN,0.50,2,Ponderosa Pine,0.101874,0.010344,0.009835,0.000999,0.775889,0.124430


,Method,Balancing_Strength,Delta_Component_mean,Delta_Component_sd
0,CGAN,0.25,0.038689,0.002066
1,CGAN,0.50,0.056223,0.003002
2,CGAN,0.75,0.066227,0.003537
3,CGAN,1.00,0.072695,0.003882
4,CWGAN-GP,0.25,0.028406,0.000694
5,CWGAN-GP,0.50,0.041278,0.001009
6,CWGAN-GP,0.75,0.048623,0.001188
7,CWGAN-GP,1.00,0.053372,0.001304
